In [4]:
"""
Импорты и переменные окружения
`load_dotenv()` подтягивает `LLM_API_KEY` из `.env`, без него RAGAS-метрики
(ячейка 5) не смогут поднять LLM-клиент. `json` и `Path` нужны только в 
ячейке 5 для записи итогового файла с метриками.
"""
import json
from pathlib import Path
import sys

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
""" golden-датасет из 10 эталонных вопросов
Структура: 7 английских вопросов по 3 темам корпуса (linear model / tee
/ model_evaluation), 2 русских для проверки мультиязычного retrieval`a,
1 мета вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в ячейке 4: считаем hit, если URl retrieved-чанка содержит
ожидаемое ключевое слово
"""

GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [7]:
""" Прямое подключение к RAG-pipeline без HTTP-слоя

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в том же процессе. Прямой вызов даёт чистые тайминги и доступ к `retrieved_contexts`,
которые понадобятся RAGAS в Ячейке 5.
"""
task_api_path = Path.cwd().parent  # поднимаемся из notebooks в task-api
sys.path.append(str(task_api_path))

from app.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

c:\Users\gavri\anaconda3\envs\task-api\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7368.37it/s]


In [12]:
""" Recall@k для retriever`a

Прогоняем каждый golden вопрос через retriever, забираем URL`ы top-k
чанков и считаем hot по keyword-совпадению. Дополнительно копим 
`retrieved_contexts` - они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], excepted_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.
       Используется в Recall@k: keyword-matching на уровне модуля sklearn
       (`linear_model` / `tree` / `model_evaluation`) - этого достаточно
       для понимания: не промазал ли retriever мимо темы целиком.
    """
    if not excepted_keywords: #Ловушка под OOC вопросы
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in excepted_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(
            urls, item["ground_truth_url_keywords"]
        )
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (10/10)


In [13]:
""" generation-метрики через RAGAS.

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и `LangchainEmbeddingsWrapper`
соответственно - без этого `evaluate()` не поймет, как через них
ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу)
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

#прогоняем chain и собираем ответы
for r in results:
    r["response"] = chain.invoke(r["question"])

#RAGAS 0.2.x требует именно такие имена полей в Dataset
ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

#LLM и эмбеддер для RAGAS оборачиваются в врапперы
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True}
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "c:\Users\gavri\anaconda3\envs\task-api\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x00000259E3C1CBC0> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-23' coro=<_async_in_context.<locals>.run_in_context() running at c:\Users\gavri\anaconda3\envs\task-api\Lib\site-packages\ipykernel\utils.py:60> wait_for=<Task pending name='Task-40' coro=<Kernel.shell_main() running at c:\Users\gavri\anaconda3\envs\task-api\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at c:\Users\gavri\anaconda3\envs\task-api\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
c:\Users\gavri\anaconda3\envs\task-api\Lib\site-packages\pydan

{'faithfulness': 0.8738, 'answer_relevancy': 0.8501}


In [16]:
"""Сохраняем сводку метрик в JSON.

Берем mean по каждой RAGAS метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, эмбеддер, топ-к). Файл `rag_metrics.json`
цитируется в README
"""
#RAGAS 0.2.x возвращает EvaluationResult - берем .to_pandas() для агрегации
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top-k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
Path("rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_questions": 10,
  "model": "meta-llama/llama-3.3-70b-instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top-k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 10
  },
  "generation": {
    "faithfulness": 0.8738095238095238,
    "answer_relevancy": 0.8500797013980043
  }
}
